# Jacobian lens — walkthrough

## 0. Setup

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [4]:
!git clone https://github.com/anthropics/jacobian-lens.git
%cd jacobian-lens

!pip install -e .

c:\Users\jason\CS\jacobian-lens\jacobian-lens\jacobian-lens\jacobian-lens


Cloning into 'jacobian-lens'...


Obtaining file:///C:/Users/jason/CS/jacobian-lens/jacobian-lens/jacobian-lens/jacobian-lens
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for jlens (pyproject.toml): started
  Building editable for jlens (pyproject.toml): finished with status 'done'
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8961 sha256=60214b08cbcc7e755796f323b6200a7edbc4901f1bca1120d2c92583842b0387
  Stored in directory: C:\Users\jason\AppData\Local\Temp\pip-ephem-wheel-cache-6gysbg3g\wheels\b0\77\db\4c455b5b3fc9b2d5c0

In [5]:
import torch
print(torch.__version__)
print(torch.__file__)

import jlens
jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

2.14.0+xpu
c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\torch\__init__.py


## 0.1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface. (Note: .to("xpu") is used to run Qwen model on Intel Arc B580. Nvidia GPU's should use .cuda() instead.)

In [ ]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to("xpu")

print(torch.xpu.memory_allocated())

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 2915.37it/s]


HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

## 0.2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [ ]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 22.12it/s]


JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

## 0.3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [ ]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

RuntimeError: level_zero backend failed with error: 40 (UR_RESULT_ERROR_OUT_OF_RESOURCES)

: 